# Mt. Hood Infrastructure Exposure Assessment

Ties your Tephra2-simulated ash hazard (`max_ash_thickness.tif`, from `tephra2_grid_analysis.ipynb`) to real
infrastructure -- roads, electric transmission lines, and regulated facilities -- near Government Camp,
Rhododendron, and Parkdale. This is the same hazard-threshold GIS framework as your Methods Report draft
("Tephra Hazard Exposure Assessment for Mt. Hood, Oregon"), with two differences:

- **Hazard source**: your own Tephra2 model output (kg/m² ash loading) instead of the published USGS hazard
  maps -- this is the piece that actually showcases your modeling work on the poster.
- **Thresholds**: the Wilson et al. (2014) / Jenkins et al. (2015) kg/m² thresholds already used throughout
  this project, instead of USGS's ≥1mm/≥10mm ashfall thresholds.

Everything else follows your Methods Report's described methodology directly: a buffer around each community,
reclassify the hazard layer into threshold polygons, intersect with infrastructure, and tabulate exposed
length (roads/lines) and facility counts per threshold per community.

### Data sources and a known unknown
Infrastructure data is pulled live from public REST APIs (Oregon/USDOT road data, BPA transmission lines, EPA
FRS facilities) filtered to a small radius around the 3 communities -- no full statewide downloads. **These
exact endpoints haven't been live-tested** (built from documentation/search, not a working test call), so the
fetch cells are written to fail loudly with a clear reason rather than silently returning nothing. If a source
doesn't work as written, every fetch function accepts a `LOCAL_*_PATH` override -- just download that layer
yourself (shapefile/GeoJSON/GeoPackage, from the sources cited in your Methods Report or elsewhere) and point
the notebook at the file instead; nothing else in the notebook needs to change.

### Hazard thresholds
| Threshold (kg/m²) | Impact |
|---:|---|
| 1 | Transport and agriculture disruption |
| 10 | Crop damage, infrastructure disruption |
| 100 | Roof collapse risk |
| 1,000 | Severe structural damage |

Wilson, T.M. et al. (2014); Jenkins, S.F. et al. (2015).

## Section 1 — Setup

Requires `max_ash_thickness.tif` (from `tephra2_grid_analysis.ipynb`'s Section 7) to already exist in the
working directory.

In [ ]:
import subprocess
subprocess.run('pip install --quiet geopandas requests', shell=True)

import os
import json

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import rasterio
from rasterio.features import shapes as raster_shapes
from shapely.geometry import Point, shape as shapely_shape
import matplotlib.pyplot as plt


In [ ]:
HAZARD_THRESHOLDS = [1, 10, 100, 1000]  # kg/m^2 -- Wilson et al. (2014), Jenkins et al. (2015)
UTM10N_EPSG = 32610   # WGS84 / UTM Zone 10N -- matches max_ash_thickness.tif and the rest of this project
WGS84_EPSG = 4326

BUFFER_RADIUS_M = 5_000  # 5 km, matching the Methods Report's community buffer

poi_names = ["Rhododendron", "Parkdale", "Govt. Camp"]
poi_locations = {
    "Rhododendron": (45.329563, -121.911191),
    "Parkdale": (45.519839, -121.596742),
    "Govt. Camp": (45.1808, -121.4509),
}

GEOTIFF_PATH = "max_ash_thickness.tif"


In [ ]:
poi_gdf = gpd.GeoDataFrame(
    {"name": list(poi_locations.keys())},
    geometry=[Point(lon, lat) for lat, lon in poi_locations.values()],
    crs=f"EPSG:{WGS84_EPSG}",
).to_crs(epsg=UTM10N_EPSG)

# 5 km buffer around each community centroid (matches the Methods Report's "4.2 Spatial Processing")
poi_buffers = poi_gdf.copy()
poi_buffers["geometry"] = poi_buffers.geometry.buffer(BUFFER_RADIUS_M)

print(f"Buffered {len(poi_buffers)} communities at {BUFFER_RADIUS_M/1000:.0f} km radius (CRS: EPSG:{UTM10N_EPSG})")
poi_buffers


## Section 2 — Infrastructure Data

One cell per source. Each tries its REST API first, filtered to a bounding box around all 3 communities'
buffers combined; if `LOCAL_*_PATH` is set (or the API call fails), it loads that local file instead via
`geopandas.read_file()`, which handles shapefile/GeoJSON/GeoPackage transparently.

In [ ]:
# Combined bounding box (WGS84 lon/lat) covering all 3 community buffers, for the API queries below
bbox_wgs84 = poi_buffers.to_crs(epsg=WGS84_EPSG).total_bounds  # [minx, miny, maxx, maxy]
print(f"Query bounding box (lon/lat): {bbox_wgs84}")


### Roads

Tries the USDOT ARNOLD (All Roads Network of Linear Referenced Data) Oregon FeatureServer first. If that
layer ID/URL is wrong, or Oregon GEOHub's "All Public Roads" dataset is preferred instead, edit
`ROADS_FEATURE_SERVER_URL` below to whatever FeatureServer/layer URL you find at
[Oregon GEOHub](https://geohub.oregon.gov/datasets/oregon-geo::all-public-roads/about) or
[ODOT's GIS portal](https://www.oregon.gov/odot/data/pages/gis%20data.aspx).

In [ ]:
ROADS_FEATURE_SERVER_URL = "https://geo.dot.gov/server/rest/services/Hosted/ARNOLD_OR_2021/FeatureServer/0"
LOCAL_ROADS_PATH = None  # e.g. "roads.geojson" -- set this to use a manually downloaded file instead


def fetch_arcgis_features(feature_server_url, bbox_wgs84, out_fields="*"):
    """Query an Esri ArcGIS FeatureServer/MapServer layer for features intersecting a WGS84 bbox."""
    query_url = feature_server_url.rstrip("/") + "/query"
    params = {
        "where": "1=1",
        "outFields": out_fields,
        "geometry": ",".join(str(v) for v in bbox_wgs84),
        "geometryType": "esriGeometryEnvelope",
        "inSR": WGS84_EPSG,
        "spatialRel": "esriSpatialRelIntersects",
        "outSR": WGS84_EPSG,
        "f": "geojson",
    }
    response = requests.get(query_url, params=params, timeout=60)
    response.raise_for_status()
    geojson = response.json()
    if "error" in geojson:
        raise RuntimeError(f"ArcGIS service error: {geojson['error']}")
    return gpd.GeoDataFrame.from_features(geojson["features"], crs=f"EPSG:{WGS84_EPSG}")


def load_roads():
    if LOCAL_ROADS_PATH:
        return gpd.read_file(LOCAL_ROADS_PATH).to_crs(epsg=UTM10N_EPSG)
    try:
        gdf = fetch_arcgis_features(ROADS_FEATURE_SERVER_URL, bbox_wgs84)
        print(f"Fetched {len(gdf)} road features from {ROADS_FEATURE_SERVER_URL}")
        return gdf.to_crs(epsg=UTM10N_EPSG)
    except Exception as exc:
        print(f"Roads API fetch failed: {exc}")
        print("Set LOCAL_ROADS_PATH to a manually downloaded roads file (shapefile/GeoJSON/GeoPackage) and re-run.")
        return gpd.GeoDataFrame(geometry=[], crs=f"EPSG:{UTM10N_EPSG}")


roads_gdf = load_roads()
roads_gdf.head()


### Electric transmission lines

Bonneville Power Administration (BPA) transmission line ArcGIS FeatureServer.

In [ ]:
TRANSMISSION_FEATURE_SERVER_URL = (
    "https://services3.arcgis.com/Iz3chmSt4P7oOoZy/arcgis/rest/services/BPA_TransmissionLines/FeatureServer/0"
)
LOCAL_TRANSMISSION_PATH = None  # e.g. "transmission_lines.geojson"


def load_transmission_lines():
    if LOCAL_TRANSMISSION_PATH:
        return gpd.read_file(LOCAL_TRANSMISSION_PATH).to_crs(epsg=UTM10N_EPSG)
    try:
        gdf = fetch_arcgis_features(TRANSMISSION_FEATURE_SERVER_URL, bbox_wgs84)
        print(f"Fetched {len(gdf)} transmission line features from {TRANSMISSION_FEATURE_SERVER_URL}")
        return gdf.to_crs(epsg=UTM10N_EPSG)
    except Exception as exc:
        print(f"Transmission lines API fetch failed: {exc}")
        print("Set LOCAL_TRANSMISSION_PATH to a manually downloaded file and re-run.")
        return gpd.GeoDataFrame(geometry=[], crs=f"EPSG:{UTM10N_EPSG}")


transmission_gdf = load_transmission_lines()
transmission_gdf.head()


### Regulated facilities

EPA Facility Registry Service (FRS) `get_facilities` REST endpoint, queried per POI with a radius search
(radius must be in miles for this API; converted from `BUFFER_RADIUS_M`).

In [ ]:
FRS_BASE_URL = "https://ofmpub.epa.gov/frs_public2/frs_rest_services.get_facilities"
LOCAL_FACILITIES_PATH = None  # e.g. "facilities.geojson"


def fetch_frs_facilities(latitude, longitude, radius_miles):
    params = {
        "output": "JSON",
        "latitude83": latitude,
        "longitude83": longitude,
        "search_radius": radius_miles,
    }
    response = requests.get(FRS_BASE_URL, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    records = data.get("Results", {}).get("FRSFacility", data) if isinstance(data, dict) else data
    if not records:
        return gpd.GeoDataFrame(geometry=[], crs=f"EPSG:{WGS84_EPSG}")

    rows = []
    for rec in records:
        lat = rec.get("Latitude83") or rec.get("latitude83")
        lon = rec.get("Longitude83") or rec.get("longitude83")
        if lat is None or lon is None:
            continue
        rows.append({**rec, "geometry": Point(float(lon), float(lat))})
    return gpd.GeoDataFrame(rows, crs=f"EPSG:{WGS84_EPSG}")


def load_facilities():
    if LOCAL_FACILITIES_PATH:
        return gpd.read_file(LOCAL_FACILITIES_PATH).to_crs(epsg=UTM10N_EPSG)

    radius_miles = BUFFER_RADIUS_M / 1609.34
    all_facilities = []
    try:
        for name, (lat, lon) in poi_locations.items():
            gdf = fetch_frs_facilities(lat, lon, radius_miles)
            gdf["queried_for"] = name
            all_facilities.append(gdf)
            print(f"Fetched {len(gdf)} facilities near {name}")
        combined = gpd.GeoDataFrame(pd.concat(all_facilities, ignore_index=True), crs=f"EPSG:{WGS84_EPSG}")
        return combined.drop_duplicates(subset="geometry").to_crs(epsg=UTM10N_EPSG)
    except Exception as exc:
        print(f"EPA FRS API fetch failed: {exc}")
        print("Set LOCAL_FACILITIES_PATH to a manually downloaded file and re-run.")
        return gpd.GeoDataFrame(geometry=[], crs=f"EPSG:{UTM10N_EPSG}")


facilities_gdf = load_facilities()
facilities_gdf.head()


## Section 3 — Hazard Threshold Zones

Reclassifies `max_ash_thickness.tif` into a binary mask at each threshold and converts each mask into
dissolved AOI polygons -- the same "reclassify -> binary polygon -> dissolve" approach as the Methods
Report's "4.1 Hazard Threshold Framework", just applied to your Tephra2 GeoTIFF instead of a USGS raster.

In [ ]:
def build_threshold_polygons(geotiff_path, thresholds):
    """Return {threshold: dissolved shapely polygon (or None if the threshold isn't reached anywhere)}."""
    with rasterio.open(geotiff_path) as src:
        band = src.read(1)
        transform = src.transform
        raster_crs = src.crs
        nodata = src.nodata

    valid = band != nodata if nodata is not None else np.isfinite(band)

    polygons = {}
    for threshold in thresholds:
        mask = valid & (band >= threshold)
        if not mask.any():
            polygons[threshold] = None
            continue
        geoms = [shapely_shape(geom) for geom, value in raster_shapes(mask.astype("uint8"), mask=mask, transform=transform)
                 if value == 1]
        polygons[threshold] = gpd.GeoSeries(geoms, crs=raster_crs).union_all()
    return polygons, raster_crs


threshold_polygons, raster_crs = build_threshold_polygons(GEOTIFF_PATH, HAZARD_THRESHOLDS)
for threshold, poly in threshold_polygons.items():
    area_km2 = poly.area / 1e6 if poly is not None else 0.0
    print(f"{threshold:>5} kg/m^2 AOI: {area_km2:,.1f} km^2")


## Section 4 — Exposure Quantification

For each community's 5 km buffer, clip roads/transmission lines to the buffer and intersect with each
threshold's AOI polygon to get exposed length (km); for facilities, count how many buffered points fall
inside each threshold's AOI. Mirrors the Methods Report's "4.3 Exposure Quantification" exactly.

In [ ]:
def exposed_length_km(lines_gdf, buffer_geom, aoi_polygon):
    if aoi_polygon is None or lines_gdf.empty:
        return 0.0
    clipped = lines_gdf.clip(buffer_geom)
    if clipped.empty:
        return 0.0
    exposed = clipped.intersection(aoi_polygon)
    return exposed.length.sum() / 1000.0


def exposed_facility_count(facilities_gdf, buffer_geom, aoi_polygon):
    if aoi_polygon is None or facilities_gdf.empty:
        return 0
    clipped = facilities_gdf.clip(buffer_geom)
    if clipped.empty:
        return 0
    return int(clipped.within(aoi_polygon).sum())


rows = []
for _, poi_row in poi_buffers.iterrows():
    name = poi_row["name"]
    buffer_geom = poi_row.geometry
    for threshold in HAZARD_THRESHOLDS:
        aoi_polygon = threshold_polygons[threshold]
        rows.append({
            "community": name,
            "threshold_kg_m2": threshold,
            "exposed_road_km": exposed_length_km(roads_gdf, buffer_geom, aoi_polygon),
            "exposed_transmission_km": exposed_length_km(transmission_gdf, buffer_geom, aoi_polygon),
            "exposed_facility_count": exposed_facility_count(facilities_gdf, buffer_geom, aoi_polygon),
        })

exposure_summary = pd.DataFrame(rows)
exposure_summary.to_csv("infrastructure_exposure_summary.csv", index=False)
print("Wrote infrastructure_exposure_summary.csv")
exposure_summary


## Section 5 — Export for QGIS

Exports the exposed infrastructure segments/points (classified by the highest threshold each one meets) and
the threshold AOI polygons themselves as GeoPackage layers -- load these alongside `max_ash_thickness.tif` in
QGIS for the poster figure, styled the same way as your Methods Report's cartographic design (hierarchical
symbology, hillshade base layer, semi-transparent AOI fills).

In [ ]:
def highest_threshold_met(geometry, thresholds, threshold_polygons, predicate):
    met = [t for t in thresholds if threshold_polygons[t] is not None and predicate(geometry, threshold_polygons[t])]
    return max(met) if met else None


def classify_by_threshold(gdf, thresholds, threshold_polygons, geom_predicate):
    if gdf.empty:
        gdf = gdf.copy()
        gdf["max_threshold_kg_m2"] = pd.Series(dtype="float")
        return gdf
    gdf = gdf.copy()
    gdf["max_threshold_kg_m2"] = gdf.geometry.apply(
        lambda geom: highest_threshold_met(geom, thresholds, threshold_polygons, geom_predicate)
    )
    return gdf[gdf["max_threshold_kg_m2"].notna()]


roads_classified = classify_by_threshold(
    roads_gdf, HAZARD_THRESHOLDS, threshold_polygons, lambda geom, poly: geom.intersects(poly))
transmission_classified = classify_by_threshold(
    transmission_gdf, HAZARD_THRESHOLDS, threshold_polygons, lambda geom, poly: geom.intersects(poly))
facilities_classified = classify_by_threshold(
    facilities_gdf, HAZARD_THRESHOLDS, threshold_polygons, lambda geom, poly: geom.within(poly))

aoi_gdf = gpd.GeoDataFrame(
    {"threshold_kg_m2": [t for t, p in threshold_polygons.items() if p is not None]},
    geometry=[p for p in threshold_polygons.values() if p is not None],
    crs=raster_crs,
)

output_gpkg = "infrastructure_exposure.gpkg"
if len(roads_classified):
    roads_classified.to_file(output_gpkg, layer="exposed_roads", driver="GPKG")
if len(transmission_classified):
    transmission_classified.to_file(output_gpkg, layer="exposed_transmission_lines", driver="GPKG")
if len(facilities_classified):
    facilities_classified.to_file(output_gpkg, layer="exposed_facilities", driver="GPKG")
aoi_gdf.to_file(output_gpkg, layer="hazard_threshold_aoi", driver="GPKG")

print(f"Wrote {output_gpkg} (CRS: EPSG:{UTM10N_EPSG}) with layers: "
      f"exposed_roads, exposed_transmission_lines, exposed_facilities, hazard_threshold_aoi")
